# Check-in 1 evidence notebook

**Goal:** re-check every number that our spine workbook (`docs/Spine_Checkin1_v0.3.xlsx`) quotes from the data,
so that nothing we show faculty on Monday is a guess.

**Scope (on purpose small):** we only verify the two journeys (trade `TXN100000380`, security `SEC937640`)
and the corporate-action / FX claims. The full defect hunt starts after Check-in 1.

**How to run:** open this notebook from the repo root folder, pick a Python 3 kernel that has `duckdb` and
`pandas` installed (`pip install duckdb pandas`), then *Run All*. It takes well under a minute.

**Why DuckDB?** DuckDB is a small database engine that runs inside Python. It can run SQL directly on the
`.csv.gz` files, so we never have to load 1.4 million rows into Excel or even into memory all at once.

## Step 1. Connect and register the ten tables as *views*

A **view** is a saved query with a name. Writing `CREATE VIEW transactions AS SELECT * FROM read_csv(...)`
means that later we can simply write `FROM transactions` and DuckDB reads the file for us.

`all_varchar=true` loads **every column as text**. The README insists on this: we convert types on purpose
later, so that a bad value (for example a date in the wrong format) is *seen* instead of silently becoming
empty.

In [1]:
import duckdb          # the SQL engine
import pandas as pd    # tables in Python (DuckDB results convert to pandas with .df())
from pathlib import Path

# ROOT = the repo folder. The notebook lives in notebooks/, so the repo is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

con = duckdb.connect()   # an in-memory database: nothing is written to disk

TABLES = ["accounts", "securities", "transactions", "corporate_actions", "pipeline_events",
          "positions", "comparison_positions", "prices", "fx_rates", "ledger_nav"]

for t in TABLES:
    path = (DATA / f"{t}.csv.gz").as_posix()
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_csv('{path}', all_varchar=true)")

# A small helper so each query below is one short line: q("SELECT ...") returns a pandas table.
def q(sql):
    return con.sql(sql).df()

print("Registered views:", ", ".join(TABLES))

Registered views: accounts, securities, transactions, corporate_actions, pipeline_events, positions, comparison_positions, prices, fx_rates, ledger_nav


## Step 2. Row counts must match `manifest.csv`

This is the first control of any pipeline: **did everything arrive?** We count the rows of each view and
compare them with the counts the organisers published in the manifest.

In [2]:
manifest = pd.read_csv(ROOT / "manifest.csv")[["table", "rows"]]

# Build one SQL statement: SELECT 'accounts' AS table, COUNT(*) FROM accounts UNION ALL SELECT ...
union_sql = " UNION ALL ".join(f"SELECT '{t}' AS table, COUNT(*) AS rows_loaded FROM {t}" for t in TABLES)
counts = q(union_sql)

check = manifest.merge(counts, on="table", how="outer")      # outer join: a missing table would show up
check["match"] = check["rows"] == check["rows_loaded"]
check

,table,rows,rows_loaded,match
0,accounts,100000,100000,True
1,comparison_positions,480183,480183,True
2,corporate_actions,200,200,True
3,fx_rates,180,180,True
4,ledger_nav,2000,2000,True
5,pipeline_events,1409533,1409533,True
6,positions,480663,480663,True
7,prices,539970,539970,True
8,securities,6012,6012,True
9,transactions,1409533,1409533,True


## Step 3. Journey 1, hop 1: the trade itself

We pull the one transaction and test the checks the spine says it passes:
`quantity x price = gross_amount`, settlement after trade date, and the ID is unique.

In [3]:
TXN = "TXN100000380"
trade = q(f"SELECT * FROM transactions WHERE transaction_id = '{TXN}'")
print("rows with this transaction_id:", len(trade))     # 1 = unique
trade.T                                                  # .T flips the table so it is easier to read

rows with this transaction_id: 1


,0
transaction_id,TXN100000380
account_id,ACC6309061
security_id,SEC937640
cusip,HAQBJOMG7
transaction_type,BUY
trade_date,2026-06-01
settlement_date,2026-06-02
quantity,1612.0
price,254.8034
currency,USD


In [4]:
r = trade.iloc[0]                                       # the first (only) row
qty, px, gross = float(r.quantity), float(r.price), float(r.gross_amount)
print(f"quantity x price = {qty * px:,.2f}   reported gross = {gross:,.2f}   difference = {qty * px - gross:.4f}")
print("trade_date:", r.trade_date, " settlement_date:", r.settlement_date)

quantity x price = 410,743.08   reported gross = 410,743.08   difference = 0.0008
trade_date: 2026-06-01  settlement_date: 2026-06-02


## Step 4. Hop 2: the pipeline timestamps

`pipeline_events` has one row per transaction with the time it was created at source (event), received by
the platform, booked, and published, plus the cutoff. We turn the text into real timestamps with
`TRY_CAST` (it returns NULL instead of crashing on a bad value) and compute the gaps in minutes.

In [5]:
pipe = q(f'''
SELECT transaction_id, source_system, batch_id,
       event_timestamp, received_timestamp, booked_timestamp, published_timestamp, cutoff_timestamp,
       date_diff('minute', TRY_CAST(event_timestamp    AS TIMESTAMP), TRY_CAST(received_timestamp  AS TIMESTAMP)) AS feed_latency_min,
       date_diff('minute', TRY_CAST(received_timestamp AS TIMESTAMP), TRY_CAST(booked_timestamp    AS TIMESTAMP)) AS booking_min,
       date_diff('minute', TRY_CAST(published_timestamp AS TIMESTAMP), TRY_CAST(cutoff_timestamp   AS TIMESTAMP)) AS minutes_before_cutoff
FROM pipeline_events WHERE transaction_id = '{TXN}'
''')
pipe.T

,0
transaction_id,TXN100000380
source_system,OMS_FRONT
batch_id,B2026060109
event_timestamp,2026-06-01 09:03:00
received_timestamp,2026-06-01 09:24:00
booked_timestamp,2026-06-01 09:28:00
published_timestamp,2026-06-01 09:33:00
cutoff_timestamp,2026-06-01 18:00:00
feed_latency_min,21
booking_min,4


## Step 5. Hop 3: master data (account and security)

The account must be OPEN on the trade date, and the trade's CUSIP and currency must match the security master.

In [6]:
acct = q(f"SELECT * FROM accounts   WHERE account_id  = '{r.account_id}'")
sec  = q(f"SELECT * FROM securities WHERE security_id = '{r.security_id}'")
display(acct.T)
display(sec.T)
print("CUSIP matches master:   ", r.cusip == sec.cusip[0])
print("currency matches master:", r.currency == sec.currency[0])

,0
account_id,ACC6309061
account_name,Synthetic Portfolio 76230
fund_id,FND00150
share_class,A
account_type,SEPARATE_ACCOUNT
base_currency,EUR
domicile,LU
strategy,INDEX
custodian,CUST_MERIDIAN
open_date,2017-08-30


,0
security_id,SEC937640
cusip,HAQBJOMG7
isin,USHAQBJOMG73
ticker,WSCL
security_name,Synthetic Issuer 3464 Common Stock
asset_class,EQUITY
currency,USD
exchange_mic,XNYS
issuer_country,GB
coupon_rate,None


CUSIP matches master:    True
currency matches master: True


## Step 6. Hop 4 and 5: rebuild the position and compare with our book and the custodian

The Dataset Guide's benchmark: start from **zero**, add BUY quantities, subtract SELL quantities, use
**BOOKED** trades only. Dates can be ISO (`2026-06-01`) or slash (`01/06/2026`, day first), so we try both
formats with `COALESCE(TRY_STRPTIME(...), TRY_STRPTIME(...))`: COALESCE takes the first value that is not NULL.

The window function `SUM(...) OVER (ORDER BY ...)` gives a **running total**, so we can see the holding
after every trade.

In [7]:
rebuild = q(f'''
WITH t AS (
  SELECT transaction_id, transaction_type, status, CAST(quantity AS DOUBLE) AS qty,
         COALESCE(TRY_STRPTIME(trade_date, '%Y-%m-%d'), TRY_STRPTIME(trade_date, '%d/%m/%Y'))::DATE AS trade_dt
  FROM transactions
  WHERE account_id = '{r.account_id}' AND security_id = '{r.security_id}'
)
SELECT *, CASE WHEN transaction_type = 'BUY' THEN qty ELSE -qty END AS signed_qty,
       SUM(CASE WHEN status = 'BOOKED' THEN (CASE WHEN transaction_type = 'BUY' THEN qty ELSE -qty END) ELSE 0 END)
           OVER (ORDER BY trade_dt, transaction_id) AS running_booked_total
FROM t ORDER BY trade_dt, transaction_id
''')
rebuild

,transaction_id,transaction_type,status,qty,trade_dt,signed_qty,running_booked_total
0,TXN100000380,BUY,BOOKED,1612.0,2026-06-01,1612.0,1612.0
1,TXN100283707,SELL,BOOKED,4034.0,2026-06-09,-4034.0,-2422.0
2,TXN100602002,BUY,BOOKED,785.0,2026-06-17,785.0,-1637.0
3,TXN101042068,BUY,BOOKED,3665.0,2026-07-01,3665.0,2028.0


In [8]:
book = q(f"SELECT * FROM positions            WHERE account_id='{r.account_id}' AND security_id='{r.security_id}'")
cust = q(f"SELECT * FROM comparison_positions WHERE account_id='{r.account_id}' AND security_id='{r.security_id}'")
booked = rebuild[rebuild.status == "BOOKED"]
print("rebuild quantity:", booked.signed_qty.sum(), "  from", len(booked), "BOOKED trades")
print("book (positions):", book.quantity.tolist(), " transaction_count:", book.transaction_count.tolist())
print("custodian       :", cust.quantity.tolist())
print("lowest running total:", rebuild.running_booked_total.min())

rebuild quantity: 2028.0   from 4 BOOKED trades
book (positions): ['2028.0']  transaction_count: ['4']
custodian       : ['2028.0']
lowest running total: -2422.0


## Step 7. Hop 6: prices and FX on the valuation date

Three vendors quote each security. The **spread** = (max - min) / mean. Our draft threshold is 0.5%.

In [9]:
px = q(f'''SELECT vendor, vendor_priority, CAST(price AS DOUBLE) AS price, currency, price_timestamp
           FROM prices WHERE security_id = '{r.security_id}' AND price_date = '2026-07-10' ORDER BY vendor_priority''')
display(px)
print(f"vendor spread = {(px.price.max() - px.price.min()) / px.price.mean():.4%}")

fx_usd = q('''SELECT MIN(CAST(rate_to_usd AS DOUBLE)) AS min_rate, MAX(CAST(rate_to_usd AS DOUBLE)) AS max_rate,
                     COUNT(*) AS days, SUM(CASE WHEN CAST(rate_to_usd AS DOUBLE) <> 1 THEN 1 ELSE 0 END) AS days_not_one
              FROM fx_rates WHERE currency = 'USD' ''')
display(fx_usd)
print("USD rate on 2026-07-10:", q("SELECT rate_to_usd FROM fx_rates WHERE currency='USD' AND rate_date='2026-07-10'").iloc[0, 0])

,vendor,vendor_priority,price,currency,price_timestamp
0,VENDOR_A,1,248.5198,USD,2026-07-10 17:00:00
1,VENDOR_B,2,248.7757,USD,2026-07-10 18:00:00
2,VENDOR_C,3,249.6191,USD,2026-07-10 19:00:00


vendor spread = 0.4415%


,min_rate,max_rate,days,days_not_one
0,0.990218,1.009784,30,30.0


USD rate on 2026-07-10: 0.995917


## Step 8. Hop 7: the fund NAV the position feeds

NAV arithmetic check: `net_assets / shares_outstanding` should equal `nav_per_share`.

In [10]:
nav = q(f'''SELECT *, CAST(net_assets AS DOUBLE) / CAST(shares_outstanding AS DOUBLE) AS recomputed_nav
            FROM ledger_nav WHERE fund_id = '{acct.fund_id[0]}' AND share_class = '{acct.share_class[0]}'
            AND as_of_date = '2026-07-10' ''')
nav.T

,0
ledger_id,LDG0000597
fund_id,FND00150
share_class,A
net_assets,40777112.070162
as_of_date,2026-07-10
shares_outstanding,3594046.98
nav_per_share,11.345737
recomputed_nav,11.345737


## Step 9. Journey 2: the security master record and its "blast radius"

How many trades, accounts, positions and funds depend on `SEC937640`? If this one master record were wrong,
all of them would be wrong at once.

In [11]:
SEC = "SEC937640"
blast = q(f'''
SELECT
  (SELECT COUNT(*)                   FROM transactions WHERE security_id = '{SEC}') AS trades,
  (SELECT COUNT(DISTINCT account_id) FROM transactions WHERE security_id = '{SEC}') AS accounts_trading,
  (SELECT COUNT(*) FROM positions WHERE security_id = '{SEC}' AND as_of_date = '2026-07-10') AS positions_0710,
  (SELECT COUNT(DISTINCT a.fund_id) FROM positions p JOIN accounts a USING (account_id)
     WHERE p.security_id = '{SEC}' AND p.as_of_date = '2026-07-10') AS funds_0710
''')
display(blast)
print("as_of_date values in positions:", q("SELECT as_of_date, COUNT(*) n FROM positions GROUP BY 1").to_dict("records"))

,trades,accounts_trading,positions_0710,funds_0710
0,309,107,107,94


as_of_date values in positions: [{'as_of_date': '2026-07-10', 'n': 480663}]


## Step 10. The corporate action on this security (our worked defect)

A spin-off creates a **new** security (the target). If the target is not in the security master, the
new holding cannot be booked or priced.

In [12]:
ca = q(f"SELECT * FROM corporate_actions WHERE security_id = '{SEC}'")
display(ca)
tgt = ca.target_security_id[0]
print("target in securities?          ", len(q(f"SELECT 1 FROM securities WHERE security_id='{tgt}'")) > 0)
tgt_use = q(f'''SELECT
   (SELECT COUNT(*) FROM positions            WHERE security_id = '{tgt}') AS book_positions,
   (SELECT COUNT(*) FROM comparison_positions WHERE security_id = '{tgt}') AS custodian_positions,
   (SELECT COUNT(*) FROM transactions         WHERE security_id = '{tgt}') AS trades,
   (SELECT COUNT(*) FROM prices               WHERE security_id = '{tgt}') AS price_quotes''')
tgt_use

,action_id,security_id,action_type,ex_date,effective_date,ratio,cash_rate,target_security_id
0,CA00093,SEC937640,SPINOFF,2026-06-11,2026-06-18,1.0,1.0532,SEC204363


target in securities?           False


,book_positions,custodian_positions,trades,price_quotes
0,63,63,64,0


**Correction to the spine.** The draft journey says no account holds `SEC204363`. The data says otherwise:
the unmastered target **is** held in our book and at the custodian, and it is traded, but it has **no price**.
So the defect is bigger than "the event could not be applied": positions exist in a security the master does
not know, and nothing can value them.

In [13]:
# Which securities appear in trades or positions but are NOT in the security master? (LEFT JOIN + IS NULL = "no match")
unmastered = q('''
WITH u AS (
  SELECT security_id, 'transactions' AS seen_in FROM transactions t LEFT JOIN securities s USING (security_id) WHERE s.security_id IS NULL
  UNION ALL
  SELECT security_id, 'positions'    AS seen_in FROM positions    p LEFT JOIN securities s USING (security_id) WHERE s.security_id IS NULL
)
SELECT u.security_id,
       SUM(CASE WHEN seen_in = 'transactions' THEN 1 ELSE 0 END) AS trade_rows,
       SUM(CASE WHEN seen_in = 'positions'    THEN 1 ELSE 0 END) AS position_rows,
       MAX(CASE WHEN c.action_id IS NOT NULL THEN c.action_id END) AS spinoff_that_names_it,
       (SELECT COUNT(*) FROM prices pr WHERE pr.security_id = u.security_id) AS price_quotes
FROM u LEFT JOIN corporate_actions c ON c.target_security_id = u.security_id AND c.action_type = 'SPINOFF'
GROUP BY u.security_id ORDER BY position_rows DESC
''')
print("unmastered securities:", len(unmastered),
      "| trade rows:", unmastered.trade_rows.sum(), "| position rows:", unmastered.position_rows.sum(),
      "| named by a spin-off:", unmastered.spinoff_that_names_it.notna().sum(),
      "| with any price:", (unmastered.price_quotes > 0).sum())
unmastered

unmastered securities: 15 | trade rows: 1131.0 | position rows: 1118.0 | named by a spin-off: 15 | with any price: 0


,security_id,trade_rows,position_rows,spinoff_that_names_it,price_quotes
0,SEC815744,90.0,90.0,CA00091,0
1,SEC370406,90.0,89.0,CA00009,0
2,SEC304911,86.0,86.0,CA00005,0
3,SEC738248,84.0,82.0,CA00040,0
4,SEC906315,82.0,82.0,CA00069,0
5,SEC188562,82.0,81.0,CA00015,0
6,SEC311910,78.0,77.0,CA00094,0
7,SEC475394,75.0,73.0,CA00042,0
8,SEC219998,72.0,72.0,CA00105,0
9,SEC298507,70.0,69.0,CA00082,0


## Step 11. Corporate actions across the whole file

One row per action type: how many events, how many have no target, how many have a target that is **not**
mastered, how many have `ex_date` after `effective_date` (chronology problem) and how many carry a
`cash_rate` (to test whether "a spin-off with a cash rate is unusual" is really special, or every row has one).

In [14]:
ca_profile = q('''
SELECT c.action_type,
       COUNT(*)                                                         AS events,
       SUM(CASE WHEN c.target_security_id IS NULL OR c.target_security_id = '' THEN 1 ELSE 0 END) AS no_target,
       SUM(CASE WHEN c.target_security_id <> '' AND s.security_id IS NULL THEN 1 ELSE 0 END)     AS target_not_mastered,
       SUM(CASE WHEN TRY_CAST(c.ex_date AS DATE) > TRY_CAST(c.effective_date AS DATE) THEN 1 ELSE 0 END) AS ex_after_effective,
       SUM(CASE WHEN c.cash_rate IS NOT NULL AND c.cash_rate <> '' THEN 1 ELSE 0 END)            AS has_cash_rate,
       SUM(CASE WHEN TRY_CAST(c.ex_date AS DATE) IS NULL OR TRY_CAST(c.effective_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS unparseable_dates,
       SUM(CASE WHEN src.security_id IS NULL THEN 1 ELSE 0 END)          AS source_not_mastered
FROM corporate_actions c
LEFT JOIN securities s   ON s.security_id   = c.target_security_id    -- LEFT JOIN keeps the action even if no match
LEFT JOIN securities src ON src.security_id = c.security_id
GROUP BY 1 ORDER BY 2 DESC
''')
# SUM() comes back as decimals; turn the count columns into whole numbers for readability
ca_profile = ca_profile.astype({c: int for c in ca_profile.columns if c != "action_type"})
ca_profile

,action_type,events,no_target,target_not_mastered,ex_after_effective,has_cash_rate,unparseable_dates,source_not_mastered
0,CASH_DIVIDEND,66,66,0,26,66,0,0
1,SPLIT,52,52,0,30,52,0,0
2,MERGER,31,31,0,12,31,0,0
3,REDEMPTION,28,28,0,11,28,0,0
4,SPINOFF,23,0,23,8,23,0,0


## Step 12. How many current holdings sit in a security hit by a spin-off or merger?

We count position rows on the snapshot date whose security is the **source** of a SPINOFF or MERGER event,
and the distinct funds behind those accounts. This is *exposure* (who could be affected), not a loss.

In [15]:
# GROUP BY action_type gives one row per event type, so we can see spin-offs on their own.
affected = q('''
WITH src AS (SELECT DISTINCT security_id, action_type FROM corporate_actions)
SELECT action_type, COUNT(*) AS holdings, COUNT(DISTINCT p.account_id) AS accounts, COUNT(DISTINCT a.fund_id) AS funds
FROM positions p
JOIN src USING (security_id)
LEFT JOIN accounts a USING (account_id)
WHERE p.as_of_date = '2026-07-10'
GROUP BY 1 ORDER BY 2 DESC
''').set_index("action_type")
affected

,holdings,accounts,funds
action_type,,,
CASH_DIVIDEND,5173,5008,500
SPLIT,4143,4080,500
MERGER,2458,2458,499
REDEMPTION,2203,2203,493
SPINOFF,1896,1896,494


## Step 13. Settlement lag (to set the rule in control C06)

Our draft rule says "lag <= T+2 unless there is a settlement_note". Since 28 May 2024 the US standard is
**T+1**. Let's look at what the data actually shows before choosing a threshold. `date_diff('day', ...)`
counts calendar days.

In [16]:
lag = q('''
WITH d AS (
  SELECT COALESCE(TRY_STRPTIME(trade_date, '%Y-%m-%d'),      TRY_STRPTIME(trade_date, '%d/%m/%Y'))::DATE      AS td,
         COALESCE(TRY_STRPTIME(settlement_date, '%Y-%m-%d'), TRY_STRPTIME(settlement_date, '%d/%m/%Y'))::DATE AS sd,
         settlement_note
  FROM transactions
)
SELECT date_diff('day', td, sd) AS lag_days,
       COUNT(*) AS trades,
       SUM(CASE WHEN settlement_note IS NOT NULL AND settlement_note <> '' THEN 1 ELSE 0 END) AS with_note
FROM d GROUP BY 1 ORDER BY 1
''')
lag

,lag_days,trades,with_note
0,-2,1124,0.0
1,1,730625,0.0
2,2,294198,0.0
3,3,183086,0.0
4,4,195856,0.0
5,7,845,845.0
6,9,2112,0.0
7,<NA>,1687,0.0


Calendar days mix in weekends: a Friday trade that settles Monday is "3 days" but only T+1. So we count
**business days** instead. `generate_series` lists every day between trade and settlement, and we keep only
Monday to Friday (`dayofweek` 0 = Sunday, 6 = Saturday). Holidays are ignored here (an assumption to state).

In [17]:
bd_lag = q('''
WITH d AS (
  SELECT COALESCE(TRY_STRPTIME(trade_date, '%Y-%m-%d'),      TRY_STRPTIME(trade_date, '%d/%m/%Y'))::DATE      AS td,
         COALESCE(TRY_STRPTIME(settlement_date, '%Y-%m-%d'), TRY_STRPTIME(settlement_date, '%d/%m/%Y'))::DATE AS sd,
         settlement_note
  FROM transactions
)
SELECT CASE WHEN td IS NULL OR sd IS NULL THEN 'unparseable'
            WHEN sd < td THEN 'settles before trade'
            ELSE 'T+' || (SELECT COUNT(*) FROM generate_series(td + 1, sd, INTERVAL 1 DAY) g(x)
                          WHERE dayofweek(x) NOT IN (0, 6)) END AS business_day_lag,
       COUNT(*) AS trades,
       COUNT(settlement_note) AS with_note          -- COUNT(column) skips NULLs
FROM d GROUP BY 1 ORDER BY 1
''')
bd_lag

,business_day_lag,trades,with_note
0,T+1,913711,0
1,T+2,490054,0
2,T+5,1274,845
3,T+6,442,0
4,T+7,1241,0
5,settles before trade,1124,0
6,unparseable,1687,0


In [18]:
# How many trade dates use the slash format (DD/MM/YYYY) instead of ISO?
q('''SELECT SUM(CASE WHEN trade_date      LIKE '__/__/____' THEN 1 ELSE 0 END) AS slash_trade_dates,
            SUM(CASE WHEN settlement_date LIKE '__/__/____' THEN 1 ELSE 0 END) AS slash_settlement_dates,
            COUNT(*) AS trades
     FROM transactions''')

,slash_trade_dates,slash_settlement_dates,trades
0,9398.0,0.0,1409533


## Step 14. Summary: spine claim vs data

Each row restates one number from the spine and what this notebook found.

In [19]:
sp = ca_profile.set_index("action_type")
summary = pd.DataFrame([
    ("Manifest row counts all match",        "yes",          bool(check["match"].all())),
    ("TXN100000380 unique",                   1,              len(trade)),
    ("Rebuild = book = custodian (2,028)",    2028,           (booked.signed_qty.sum(), book.quantity.tolist(), cust.quantity.tolist())),
    ("Lowest running total (zero opening)",   -2422,          rebuild.running_booked_total.min()),
    ("SEC937640 trades / accounts",           "309 / 107",    f"{blast.trades[0]} / {blast.accounts_trading[0]}"),
    ("SEC937640 positions / funds on 07-10",  "107 / 94",     f"{blast.positions_0710[0]} / {blast.funds_0710[0]}"),
    ("Spin-offs with unmastered target",      "23 of 23",     f"{sp.loc['SPINOFF','target_not_mastered'] if 'SPINOFF' in sp.index else 'n/a'} of {sp.loc['SPINOFF','events'] if 'SPINOFF' in sp.index else 'n/a'}"),
    ("Mergers with no target",                "31 of 31",     f"{sp.loc['MERGER','no_target'] if 'MERGER' in sp.index else 'n/a'} of {sp.loc['MERGER','events'] if 'MERGER' in sp.index else 'n/a'}"),
    ("Holdings / funds in spin-off sources",  "1,893 / 494",  f"{affected.loc['SPINOFF','holdings']:,} / {affected.loc['SPINOFF','funds']}"),
    ("SEC204363 not held by any account",     "not held",     f"{tgt_use.book_positions[0]} book / {tgt_use.custodian_positions[0]} custodian positions"),
    ("All events: ex_date after effective",   "(not stated)", int(ca_profile.ex_after_effective.sum())),
    ("All events carry a cash_rate",          "spin-off only", f"{int(ca_profile.has_cash_rate.sum())} of {int(ca_profile.events.sum())}"),
    ("USD rate_to_usd range",                 "0.990-1.010",  f"{fx_usd.min_rate[0]:.3f}-{fx_usd.max_rate[0]:.3f}"),
], columns=["claim", "spine says", "data shows"])
summary

,claim,spine says,data shows
0,Manifest row counts all match,yes,True
1,TXN100000380 unique,1,1
2,"Rebuild = book = custodian (2,028)",2028,"(2028.0, [2028.0], [2028.0])"
3,Lowest running total (zero opening),-2422,-2422.0
4,SEC937640 trades / accounts,309 / 107,309 / 107
5,SEC937640 positions / funds on 07-10,107 / 94,107 / 94
6,Spin-offs with unmastered target,23 of 23,23 of 23
7,Mergers with no target,31 of 31,31 of 31
8,Holdings / funds in spin-off sources,"1,893 / 494","1,896 / 494"
9,SEC204363 not held by any account,not held,63 book / 63 custodian positions
